# 01 — Mandatory three-schema preflight

F0 only. Protocol v1.5 + implementation lock v1.5k.


In [5]:
from pathlib import Path
import sys, pandas as pd
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from ad_f0.io import load_config, ensure_dirs
cfg=load_config(ROOT); ensure_dirs(ROOT,cfg)
print('ROOT =',ROOT)


ROOT = C:\Users\uysal\AD_ROUTER_CP_F0_v1.8


In [8]:
# Safety switch. This notebook is allowed to download ONLY CHEMBL02, CHEMBL09, CHEMBL17.
# Set True when you want the notebook to fetch the three official SQLite probe archives.
AUTO_DOWNLOAD_PROBES = True
RETAIN_PROBE_ARCHIVES = False

manifest=pd.read_csv(ROOT/cfg['paths']['release_manifest'],dtype={'release':str})
probe_labels=list(cfg['chembl']['schema_probe_releases'])
probe=manifest[manifest.release_label.isin(probe_labels)].copy()
assert set(probe.release_label)==set(probe_labels), f'Manifest missing probe releases: {probe_labels}'
print(probe[['release','release_label','local_db_path']].to_string(index=False))


release release_label  local_db_path
      2      CHEMBL02            NaN
      9      CHEMBL09            NaN
     17      CHEMBL17            NaN


In [9]:
from ad_f0.archive_download import download_extract_sqlite

download_rows=[]
resolved_paths={}
for _,r in probe.iterrows():
    raw='' if pd.isna(r.get('local_db_path')) else str(r.get('local_db_path')).strip()
    local=(ROOT/raw).resolve() if raw else None
    if local is not None and local.exists():
        resolved_paths[r.release_label]=local
        download_rows.append({'release_label':r.release_label,'release':r.release,'source':'preconfigured_local',
                              'url':'local','db_path':str(local),'archive_sha256':'','db_sha256':''})
        continue
    if not AUTO_DOWNLOAD_PROBES:
        continue
    got=download_extract_sqlite(
        r.release,
        ROOT/cfg['paths']['archives_dir']/'probe_archives',
        ROOT/cfg['paths']['cache_dir']/'probe_sqlite',
        retain_archive=RETAIN_PROBE_ARCHIVES,
    )
    resolved_paths[r.release_label]=got.db_path
    download_rows.append({'release_label':r.release_label,'release':r.release,'source':'official_sqlite_download',
                          'url':got.url,'db_path':str(got.db_path),'archive_sha256':got.archive_sha256,'db_sha256':got.db_sha256})

pd.DataFrame(download_rows).to_csv(ROOT/'results'/'audit'/'f0_probe_download_manifest.csv',index=False)
missing=[x for x in probe_labels if x not in resolved_paths]
if missing:
    print('Missing probe DBs:',missing)
    print('Either set local_db_path for these releases in config/chembl_release_manifest.csv or set AUTO_DOWNLOAD_PROBES=True and rerun this notebook.')
    raise RuntimeError('F0-PREFLIGHT is incomplete. Bulk download is forbidden.')
print('Probe DB paths resolved:',resolved_paths)


Probe DB paths resolved: {'CHEMBL02': WindowsPath('C:/Users/uysal/AD_ROUTER_CP_F0_v1.8/data/cache/probe_sqlite/chembl_02.db'), 'CHEMBL09': WindowsPath('C:/Users/uysal/AD_ROUTER_CP_F0_v1.8/data/cache/probe_sqlite/chembl_09.db'), 'CHEMBL17': WindowsPath('C:/Users/uysal/AD_ROUTER_CP_F0_v1.8/data/cache/probe_sqlite/chembl_17.db')}


In [10]:
from ad_f0.chembl_adapter import schema_probe_row, build_target_sequence_bridge, audit_pre9_target_bridge

rows=[schema_probe_row(resolved_paths[label],label) for label in probe_labels]
probe_df=pd.DataFrame(rows)
expected=cfg['chembl']['schema_probe_expected_generation']
for _,r in probe_df.iterrows():
    assert r.schema_generation==expected[r.release_label], (r.release_label,r.schema_generation,expected[r.release_label])
probe_df.to_csv(ROOT/'results'/'audit'/'f0_schema_probe.csv',index=False)
meta_cols=[c for c in [
    'release_label','schema_generation','linked_document_count',
    'document_year_null_rate','document_title_null_rate','document_first_page_null_rate',
    'document_weak_alias_bridgeable_rate'
] if c in probe_df.columns]
probe_df[meta_cols].to_csv(ROOT/'results'/'audit'/'f0_probe_document_metadata_completeness.csv',index=False)
display(probe_df)
print('Document metadata completeness:')
display(probe_df[meta_cols])

# Identity-only reference bridge: CHEMBL09 target metadata only, never activity evidence.
bridge=build_target_sequence_bridge(resolved_paths[cfg['chembl']['pre9_reference_bridge_release']])
bridge_path=ROOT/'data'/'interim'/'f0_pre9_target_sequence_bridge.parquet'
bridge.to_parquet(bridge_path,index=False)
assert int(bridge.bridge_status.eq('unique').sum())>0, 'CHEMBL09 sequence bridge has no unique mappings.'

pre9_audit=audit_pre9_target_bridge(resolved_paths['CHEMBL02'],bridge,cfg['chembl']['organism'])
pre9_audit.to_csv(ROOT/'results'/'audit'/'f0_pre9_target_bridge_audit.csv',index=False)
summary=(pre9_audit.groupby('bridge_status',dropna=False).size().rename('targets').reset_index())
display(summary)
resolved_unique=int(pre9_audit.bridge_status.eq('unique').sum())
assert resolved_unique>0, 'CHEMBL02 human target sequences have no unique mapping through the fixed CHEMBL09 bridge.'

assert probe_df.schema_structural_probe_pass.astype(bool).all(), 'At least one representative schema is structurally unsupported.'
assert probe_df.primary_semantics_probe_pass.astype(bool).all(), (
    'Primary target semantics are not certified in at least one schema family. '
    'In particular, pre9 confidence=7 is NOT promoted to modern confidence=9 without relationship_type=D.'
)
print('F0-PREFLIGHT PASSED. No bulk archive was downloaded by this assertion. Next: 02_F0_release_manifest.ipynb')


[16:46:15] Initializing Normalizer


,release_label,db_path,linked_document_count,document_year_null_rate,document_title_null_rate,document_first_page_null_rate,document_weak_alias_bridgeable_rate,schema_generation,assay_type_available,relationship_type_available,...,target_type_semantics_probe_pass,activity_endpoint_column,activity_relation_column,compound_smiles_available,compound_molfile_available,potential_duplicate_available,data_validity_comment_available,schema_structural_probe_pass,primary_semantics_probe_pass,schema_probe_pass
0,CHEMBL02,C:\Users\uysal\AD_ROUTER_CP_F0_v1.8\data\cache...,33956,0.000000,1.000000,0.000000,1.000000,pre9_early,True,True,...,True,standard_type,relation,True,True,False,False,True,True,True
1,CHEMBL09,C:\Users\uysal\AD_ROUTER_CP_F0_v1.8\data\cache...,39091,0.000026,0.999974,0.000026,0.999974,pre15_legacy,True,True,...,True,standard_type,relation,True,False,False,False,True,True,True
2,CHEMBL17,C:\Users\uysal\AD_ROUTER_CP_F0_v1.8\data\cache...,51271,0.000449,0.983168,0.000449,0.999551,post15,True,True,...,True,standard_type,standard_relation,True,False,True,True,True,True,True


Document metadata completeness:


,release_label,schema_generation,linked_document_count,document_year_null_rate,document_title_null_rate,document_first_page_null_rate,document_weak_alias_bridgeable_rate
0,CHEMBL02,pre9_early,33956,0.000000,1.000000,0.000000,1.000000
1,CHEMBL09,pre15_legacy,39091,0.000026,0.999974,0.000026,0.999974
2,CHEMBL17,post15,51271,0.000449,0.983168,0.000449,0.999551


,bridge_status,targets
0,unique,1925
1,unmapped,116


F0-PREFLIGHT PASSED. No bulk archive was downloaded by this assertion. Next: 02_F0_release_manifest.ipynb
